# X-Mens: decidual vs fibroblast nuclei, pseudobulk DESeq2

Which genes differ between decidual cells and endometrial fibroblasts in the induced mouse model? The nuclei of each
cell type are summed into one pseudobulk per timepoint, and PyDESeq2 compares the two cell types within timepoints, so
the five induced timepoints serve as replicates and timepoint effects stay out of the comparison. The uninduced
sample's fibroblasts form a sixth fibroblast pseudobulk; its own timepoint term fits it exactly, so it does not enter
the decidual vs fibroblast estimate directly, but it shapes the size factors, the dispersions and the expression
filter. This notebook reproduces the fit of record of 27 October 2026, `xmens_broad_all_genes_with_uninduced.csv`.

- **Model:** `~ timepoint + group`, Wald test of decidual vs fibroblast; a positive log2 fold change means higher in
  decidual nuclei. Called DE: adjusted p < 0.05 and |log2 fold change| > 1, unshrunk.
- **Inputs:** `final_mouse.h5ad` (`obs` and the raw counts in `layers/counts`) and the feature manifest, which maps
  the object's features to Ensembl 116 genes.
- **Outputs** (in `OUT_DIR`, in the record's file formats): the all-genes results table, the pseudobulk counts and
  descriptions, the gene table and the fit diagnostics.
- **Environment:** rsc_uv (PyDESeq2 0.5.4, NumPy 2.2.0). Exact agreement with the record depends on these versions.
- **Runtime:** about 3 minutes and 3.5 GB of memory on 8 CPUs. Importing the packages and checksumming the 2 GB object
  take most of it; the fit takes about 30 seconds.

In [ ]:
# Parameters; papermill -p NAME VALUE overrides them. Relative paths start from this notebook's folder.

# Inputs of record: the object and the feature manifest the fit of record read.
H5AD = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/interim/final_mouse.h5ad"
FEATURE_MANIFEST = "resources/xmens_feature_manifest.tsv"

# The table of record and its pseudobulks, compared with this run at the end; None skips that comparison.
RECORD = "record/xmens_broad_all_genes_with_uninduced.csv.gz"
RECORD_PSEUDOBULKS = "record/xmens_broad_metadata_with_uninduced.tsv"
RECORD_COUNTS = "record/xmens_broad_carrier_counts_with_uninduced.tsv.gz"

OUT_DIR = None  # None writes nothing
FORCE = False  # True lets this run replace the outputs already in OUT_DIR
N_CPUS = 8  # PyDESeq2 workers; the results are the same on 1 CPU

In [ ]:
import hashlib
import io
import json
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pydeseq2
import scipy.sparse as sp
from anndata.io import read_elem
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

print(f"PyDESeq2 {pydeseq2.__version__}, NumPy {np.__version__}, pandas {pd.__version__}")

## Settings

The settings of the fit of record, and the names of its output files.

In [ ]:
# Settings of the fit of record; changing any of them changes the results.

# Nuclei: every subtype of the two cell types, the apoptotic decidual cells (decAp) included.
LABEL_COLUMN = "coarse_refined"
ARM = {"decidual cells": "decidual", "endometrial fibroblasts": "fibroblast"}

# The 11 pseudobulks as (timepoint, arm), in the record's order: both arms at each induced timepoint, then the
# uninduced fibroblasts. The uninduced sample's 64 decidual-labeled nuclei are left out, since an uninduced uterus
# has no decidual cells to compare.
INDUCED_TIMEPOINTS = ["1_dpi_early", "1_dpi_late", "2_dpi", "3_dpi", "4_dpi_bleeding"]
PSEUDOBULK_KEYS = [(timepoint, arm) for timepoint in INDUCED_TIMEPOINTS for arm in ("decidual", "fibroblast")]
PSEUDOBULK_KEYS.append(("uninduced", "fibroblast"))
MIN_NUCLEI = 20  # per pseudobulk, as in the human and Ulrich fits; the smallest here has 1,170

# A gene is fitted if its CPM is at least MIN_CPM in at least half (rounded up) of the decidual or of the fibroblast
# pseudobulks. Each arm is checked on its own, so genes expressed in only one cell type stay in.
MIN_CPM = 1.0

DESIGN = "~ timepoint + group"

# Called DE when both hold, on the unshrunk fold change.
MAX_PADJ = 0.05
MIN_ABS_LOG2FC = 1.0

In [ ]:
# Outputs, named as in the record's folder. Checked before the fit, so a run does not end by refusing to overwrite.
OUTPUTS = {
    "table": "xmens_broad_all_genes_with_uninduced.csv",
    "counts": "xmens_broad_carrier_counts_with_uninduced.tsv",
    "pseudobulks": "xmens_broad_metadata_with_uninduced.tsv",
    "genes": "xmens_broad_genes.tsv",
    "diagnostics": "fit_diagnostics_with_uninduced.json",
}
if OUT_DIR is not None:
    existing = [name for name in OUTPUTS.values() if (Path(OUT_DIR) / name).exists()]
    assert FORCE or not existing, f"{OUT_DIR} already holds {existing}; set FORCE = True to overwrite them"

## Load

Only what the pseudobulks need is read from the 2 GB object: `obs`, the feature names and the raw UMI counts in
`layers/counts`. Each input's SHA-256 is printed and compared with that of the file the fit of record read.

In [ ]:
# SHA-256 of the files the fit of record read.
SHA256_OF_RECORD = {
    "object": "7cfa0ebda34812fbfbc9ea4fbe23888e32997ad0bc8263634abc7f9f30ad7116",
    "feature manifest": "551649d3d30bee8a5a40515bd4b3c28e0426ff0fe1c6c1bbd97816733fbb79cf",
}


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


for name, path in (("object", H5AD), ("feature manifest", FEATURE_MANIFEST)):
    checksum = sha256_of(path)
    verdict = "the file of record" if checksum == SHA256_OF_RECORD[name] else "NOT the file of record"
    print(f"{name}: {path}\n  sha256 {checksum} ({verdict})")

In [ ]:
with h5py.File(H5AD, "r") as f:
    obs = read_elem(f["obs"])
    feature_names = read_elem(f["var"]).index.astype(str)
    counts = sp.csr_matrix(read_elem(f["layers/counts"]))  # nuclei x features

# DESeq2 models raw counts: whole, non-negative numbers.
assert counts.shape == (len(obs), len(feature_names))
assert obs.index.is_unique and feature_names.is_unique
if not np.issubdtype(counts.dtype, np.integer):
    assert np.array_equal(counts.data, np.round(counts.data)), "layers/counts holds non-integer values"
assert counts.data.min() >= 0, "layers/counts holds negative values"
counts = counts.astype(np.int64)  # sums over thousands of nuclei stay exact

print(f"{counts.shape[0]:,} nuclei x {counts.shape[1]:,} features, {counts.nnz:,} nonzero counts")

In [ ]:
# Default NA parsing, as in the record: the 64 genes with an empty symbol get a missing one.
manifest = pd.read_csv(FEATURE_MANIFEST, sep="\t", dtype={"canonical_id": "string"})
manifest = manifest.sort_values("source_index", kind="stable").reset_index(drop=True)
assert manifest["source_feature_id"].astype(str).tolist() == feature_names.tolist(), \
    "the manifest does not list the object's features in the object's order"

# Features the manifest excludes (absent from its gene annotation) or that have no Ensembl gene are dropped.
is_kept = ~manifest["excluded"] & manifest["canonical_id"].notna()
kept = manifest[is_kept]

# Every kept feature is its own gene, so a gene's counts are its feature's counts. The code of record sums features
# that share a gene; none do in this manifest, and the assertion stops the notebook if one ever does.
assert kept["canonical_id"].is_unique, "several features map to one gene"
genes = pd.DataFrame({
    "ensembl_gene_id": kept["canonical_id"].astype(str).to_numpy(),
    "gene_symbol": kept["gene_symbol"].to_numpy(),
    "biotype": kept["biotype"].to_numpy(),
    "source_feature_ids": kept["source_feature_id"].astype(str).to_numpy(),
    "source_feature_n": 1,
})
gene_columns = kept["source_index"].to_numpy()  # each gene's column in counts

print(f"{len(genes):,} genes from {len(manifest):,} features; {(~is_kept).sum()} features dropped")

## Pseudobulks

UMI counts summed over the nuclei of one cell type at one timepoint, across the timepoint's samples. Nuclei per
timepoint and cell type:

In [ ]:
arm = obs[LABEL_COLUMN].astype(str).map(ARM)  # missing for every other cell type
timepoint = obs["timepoint"].astype(str)
pd.crosstab(timepoint[arm.notna()], arm[arm.notna()])

In [ ]:
descriptions, members = [], []
for timepoint_name, arm_name in PSEUDOBULK_KEYS:
    nuclei = np.flatnonzero((timepoint == timepoint_name).to_numpy() & (arm == arm_name).to_numpy())
    samples = obs["sample"].iloc[nuclei].astype(str)
    descriptions.append({
        "pseudobulk_id": f"timepoint={timepoint_name}__group={arm_name}",
        "timepoint": timepoint_name,
        "group": arm_name,
        "n_cells": len(nuclei),
        "n_samples": samples.nunique(),
        "samples": ";".join(sorted(samples.unique())),
    })
    members.append(nuclei)
pseudobulks = pd.DataFrame(descriptions)
assert pseudobulks["n_cells"].ge(MIN_NUCLEI).all(), pseudobulks[["pseudobulk_id", "n_cells"]]

# membership[p, n] = 1 when nucleus n is in pseudobulk p, so the product sums each pseudobulk's nuclei.
pseudobulk_of_member = np.repeat(np.arange(len(members)), [len(nuclei) for nuclei in members])
membership = sp.csr_matrix(
    (np.ones(len(pseudobulk_of_member), dtype=np.int64), (pseudobulk_of_member, np.concatenate(members))),
    shape=(len(members), counts.shape[0]),
)
pseudobulk_counts = pd.DataFrame(
    (membership @ counts).toarray()[:, gene_columns],
    index=pseudobulks["pseudobulk_id"].to_numpy(),
    columns=genes["ensembl_gene_id"].to_numpy(),
)

print(f"{pseudobulks['n_cells'].sum():,} nuclei in {len(pseudobulks)} pseudobulks")
pseudobulks

## Fit

The expression filter, then PyDESeq2 on the genes that pass. Cook's distance outliers are refit and filtered
(PyDESeq2's defaults). Independent filtering is off, so the adjusted p-values are Benjamini-Hochberg over exactly the
fitted genes.

In [ ]:
is_decidual = (pseudobulks["group"] == "decidual").to_numpy()

# CPM over all of the object's genes, not only those that pass.
cpm = pseudobulk_counts.div(pseudobulk_counts.sum(axis=1), axis=0) * 1e6
needed_decidual = (is_decidual.sum() + 1) // 2  # 3 of 5
needed_fibroblast = ((~is_decidual).sum() + 1) // 2  # 3 of 6
passes_filter = (
    cpm.loc[is_decidual].ge(MIN_CPM).sum(axis=0).ge(needed_decidual)
    | cpm.loc[~is_decidual].ge(MIN_CPM).sum(axis=0).ge(needed_fibroblast)
)

print(f"{passes_filter.sum():,} of {len(passes_filter):,} genes pass the expression filter")

In [ ]:
design_data = pd.DataFrame(
    {"timepoint": pseudobulks["timepoint"].to_numpy(), "group": pseudobulks["group"].to_numpy()},
    index=pseudobulks["pseudobulk_id"].to_numpy(),
)
inference = DefaultInference(n_cpus=N_CPUS)
dds = DeseqDataSet(
    counts=pseudobulk_counts.loc[:, passes_filter.to_numpy()],
    metadata=design_data,
    design=DESIGN,
    refit_cooks=True,
    inference=inference,
    quiet=True,
)
dds.deseq2()

# PyDESeq2 orders factor levels alphabetically, so decidual is the reference level and the design's group
# coefficient is fibroblast vs decidual. 11 pseudobulks and 7 coefficients leave 4 residual degrees of freedom.
GROUP_COEFFICIENT = "group[T.fibroblast]"
design_matrix = dds.obsm["design_matrix"]
rank = int(np.linalg.matrix_rank(design_matrix.to_numpy(dtype=float)))
assert rank == design_matrix.shape[1], f"the design {DESIGN} is rank-deficient"
assert [column for column in design_matrix.columns if column.startswith("group[T.")] == [GROUP_COEFFICIENT]

print(f"{design_matrix.shape[0]} pseudobulks, {rank} coefficients: {list(design_matrix.columns)}")

In [ ]:
stats = DeseqStats(
    dds,
    contrast=["group", "decidual", "fibroblast"],
    independent_filter=False,
    inference=inference,
    quiet=True,
)
stats.summary()
results = stats.results_df.copy()  # lfc_shrink overwrites results_df

# apeglm shrinkage of the same effect, for display only. It shrinks a design coefficient, which is fibroblast vs
# decidual, so its sign is flipped.
stats.lfc_shrink(coeff=GROUP_COEFFICIENT)
shrunk = stats.results_df["log2FoldChange"] * -1.0

# Where both fold changes are clearly away from zero they must agree in sign, or the flip would be wrong.
clear = results["log2FoldChange"].abs().gt(1) & shrunk.abs().gt(0.5)
disagree = np.sign(results.loc[clear, "log2FoldChange"]) != np.sign(shrunk[clear])
assert not (clear.any() and disagree.mean() > 0.01), "shrunk and unshrunk fold changes disagree in sign"

## Results

One row per gene of the object, in the record's columns. Genes that did not pass the filter keep their annotation
and counts, with empty statistics. A missing adjusted p-value would count as not significant.

In [ ]:
fitted = pd.DataFrame({
    "ensembl_gene_id": results.index.astype(str),
    "baseMean": results["baseMean"].to_numpy(),
    "log2FoldChange_unshrunk": results["log2FoldChange"].to_numpy(),
    "log2FoldChange_shrunk": shrunk.reindex(results.index).to_numpy(),
    "lfcSE": results["lfcSE"].to_numpy(),
    "stat": results["stat"].to_numpy(),
    "pvalue": results["pvalue"].to_numpy(),
    "padj": results["padj"].to_numpy(),
})
fitted["estimable"] = np.isfinite(fitted["log2FoldChange_unshrunk"]) & np.isfinite(fitted["pvalue"])

table = genes.copy()
table["carrier"] = True  # every gene of the object is in the table
table["production_eligible"] = passes_filter.to_numpy()  # passed the expression filter, so was fitted
table["total_count"] = pseudobulk_counts.sum(axis=0).to_numpy(dtype=np.int64)
table["nonzero_pseudobulks"] = pseudobulk_counts.gt(0).sum(axis=0).to_numpy(dtype=np.int64)
assert fitted["ensembl_gene_id"].tolist() == table.loc[table["production_eligible"], "ensembl_gene_id"].tolist()
table = table.merge(fitted, on="ensembl_gene_id", how="left", validate="one_to_one")

STATISTICS = ["log2FoldChange_unshrunk", "log2FoldChange_shrunk", "lfcSE", "stat", "pvalue", "padj"]
table["estimable"] = table["production_eligible"] & table["estimable"].astype("boolean").fillna(False).astype(bool)
table.loc[~table["estimable"], STATISTICS] = np.nan
table["padj_for_call"] = table["padj"].where(np.isfinite(table["padj"]), 1.0)
table["significant_fdr05_lfc1"] = (
    table["estimable"]
    & table["padj_for_call"].lt(MAX_PADJ)
    & table["log2FoldChange_unshrunk"].abs().gt(MIN_ABS_LOG2FC)
)

# Fixed columns of the record's format; the source ID names the design with the uninduced fibroblasts.
table["source_id"] = "mouse_all_decidual_vs_fibroblast_timepoint_blocked_with_uninduced_fibroblasts"
table["positive_lfc_meaning"] = "decidual_minus_fibroblast"
table["analysis_method"] = "pydeseq2_wald_fixed_block"
table["n_pseudobulks_numerator"] = int(is_decidual.sum())
table["n_pseudobulks_denominator"] = int((~is_decidual).sum())

In [ ]:
is_up = table["significant_fdr05_lfc1"] & table["log2FoldChange_unshrunk"].gt(0)
is_down = table["significant_fdr05_lfc1"] & table["log2FoldChange_unshrunk"].lt(0)
print(f"{table['production_eligible'].sum():,} genes tested; called DE: {is_up.sum():,} higher in decidual, "
      f"{is_down.sum():,} higher in fibroblast nuclei")
shown = ["gene_symbol", "baseMean", "log2FoldChange_unshrunk", "log2FoldChange_shrunk", "padj"]
table.loc[table["significant_fdr05_lfc1"], shown].sort_values("padj").head(10)

In [ ]:
# The record's fit diagnostics, key for key.
size_factors = dds.obs["size_factors"]
arms = ("decidual", "fibroblast")
diagnostics = {
    "design": DESIGN,
    "design_columns": list(design_matrix.columns),
    "design_rank": rank,
    "residual_df": int(design_matrix.shape[0] - rank),
    "coefficient": GROUP_COEFFICIENT,
    "n_pseudobulks": int(len(design_data)),
    "n_genes_fitted": int(len(fitted)),
    "size_factor_min": float(size_factors.min()),
    "size_factor_median": float(size_factors.median()),
    "size_factor_max": float(size_factors.max()),
    "n_padj_na": int(fitted["padj"].isna().sum()),
    "n_not_estimable": int((~fitted["estimable"]).sum()),
    "pseudobulk_design": "with_uninduced",
    "pseudobulks_per_arm": {name: int((pseudobulks["group"] == name).sum()) for name in arms},
    "nuclei_per_arm": {name: int(pseudobulks.loc[pseudobulks["group"] == name, "n_cells"].sum()) for name in arms},
    "eligible_genes": int(passes_filter.sum()),
    "strict_calls": int(table["significant_fdr05_lfc1"].sum()),
    "calls_up": int(is_up.sum()),
    "calls_down": int(is_down.sum()),
}
print(json.dumps(diagnostics, indent=1))

## Check against the record

The results table, as it is written to disk (15 significant digits), is read back like the table of record, with
`float_precision="round_trip"`. They must list the same genes, hold identical values in every column (missing equal
to missing) and call the same genes DE in the same direction. The pseudobulks must have the record's nuclei counts
and summed counts. Any difference stops the notebook here, before anything is saved; set the `RECORD` parameters to
`None` to run without the check.

In [ ]:
# The record's file formats: 15 significant digits, empty fields (TSV: NA) for missing values.
TABLE_FORMAT = {"index": False, "na_rep": "", "float_format": "%.15g", "lineterminator": "\n"}
TSV_FORMAT = {"sep": "\t", "na_rep": "NA", "float_format": "%.15g", "lineterminator": "\n"}


def read_results(source):
    """A results table read back as written: empty fields are missing and nothing else is, floats round-trip."""
    return pd.read_csv(source, keep_default_na=False, na_values=[""], float_precision="round_trip")


def compare_columns(ours, record):
    """For each column of the record: present here, identical (missing equal to missing), largest difference."""
    rows = []
    for column in record.columns:
        row = {"column": column, "present": column in ours.columns, "identical": False, "max_abs_diff": np.nan}
        if row["present"]:
            mine, theirs = ours[column], record[column]
            if pd.api.types.is_numeric_dtype(theirs) and not pd.api.types.is_bool_dtype(theirs):
                mine, theirs = mine.to_numpy(dtype=float), theirs.to_numpy(dtype=float)
                row["identical"] = np.array_equal(mine, theirs, equal_nan=True)
                both = np.isfinite(mine) & np.isfinite(theirs)
                row["max_abs_diff"] = float(np.abs(mine[both] - theirs[both]).max()) if both.any() else np.nan
            else:
                row["identical"] = mine.equals(theirs)
        rows.append(row)
    return pd.DataFrame(rows)


def calls(results_table):
    """+1 for genes called higher in decidual nuclei, -1 for lower, 0 for the rest."""
    return np.sign(results_table["log2FoldChange_unshrunk"]).where(results_table["significant_fdr05_lfc1"], 0)


table_csv = table.to_csv(**TABLE_FORMAT)
check = {}
if RECORD is not None:
    ours = read_results(io.StringIO(table_csv)).set_index("ensembl_gene_id")
    record = read_results(RECORD).set_index("ensembl_gene_id")
    assert set(ours.index) == set(record.index), "the tables list different genes"
    same_order = ours.index.equals(record.index)
    columns = compare_columns(ours, record.loc[ours.index])
    print(f"{len(ours):,} genes, the same as the record's (same order: {same_order})")
    print(columns.to_string(index=False))

    same_calls = calls(ours).equals(calls(record).loc[ours.index])
    print(f"DE calls identical: {same_calls} ({calls(ours).eq(1).sum():,} up, {calls(ours).eq(-1).sum():,} down)")
    check["table"] = bool(same_order and columns["identical"].all() and same_calls)

In [ ]:
if RECORD_PSEUDOBULKS is not None:
    record_pseudobulks = pd.read_csv(RECORD_PSEUDOBULKS, sep="\t")
    nuclei_per_pseudobulk = pseudobulks[["pseudobulk_id", "n_cells"]].merge(
        record_pseudobulks[["pseudobulk_id", "n_cells"]],
        on="pseudobulk_id",
        how="outer",
        suffixes=("", "_record"),
    )
    print(nuclei_per_pseudobulk.to_string(index=False))
    check["pseudobulks"] = bool(pseudobulks.equals(record_pseudobulks))
    print(f"pseudobulk descriptions identical: {check['pseudobulks']}")

if RECORD_COUNTS is not None:
    record_counts = pd.read_csv(RECORD_COUNTS, sep="\t", index_col="pseudobulk_id")
    same_layout = (record_counts.index.equals(pseudobulk_counts.index)
                   and record_counts.columns.equals(pseudobulk_counts.columns))
    n_different = int((record_counts.to_numpy() != pseudobulk_counts.to_numpy()).sum()) if same_layout else None
    check["counts"] = bool(same_layout and n_different == 0)
    print(f"summed counts: same pseudobulks and genes {same_layout}; entries that differ: {n_different}")

assert all(check.values()), f"the run differs from the record: {check}"

## Save

The five files of the record's folder, in its formats. Nothing is written when `OUT_DIR` is `None`.

In [ ]:
if OUT_DIR is not None:
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(out_dir / OUTPUTS["table"], **TABLE_FORMAT)
    pseudobulk_counts.to_csv(out_dir / OUTPUTS["counts"], index_label="pseudobulk_id", **TSV_FORMAT)
    pseudobulks.to_csv(out_dir / OUTPUTS["pseudobulks"], index=False, **TSV_FORMAT)
    genes.to_csv(out_dir / OUTPUTS["genes"], sep="\t", index=False, na_rep="", lineterminator="\n")
    (out_dir / OUTPUTS["diagnostics"]).write_text(json.dumps(diagnostics, indent=1))
    print(f"wrote {', '.join(OUTPUTS.values())} in {out_dir}")

## Summary

In [ ]:
outcome = "skipped" if not check else ("identical to the record" if all(check.values()) else f"differs: {check}")
print(f"genes tested:    {table['production_eligible'].sum():,}")
print(f"genes called DE: {table['significant_fdr05_lfc1'].sum():,} ({is_up.sum():,} up, {is_down.sum():,} down)")
print(f"check:           {outcome} ({', '.join(check) or 'no record given'})")